In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
from shapely.geometry import box, LineString, MultiLineString
from shapely.ops import unary_union, polygonize, linemerge
import matplotlib
import numpy as np
import warnings
import os

warnings.filterwarnings("ignore", category=UserWarning)

# =========================
# Global style
# =========================
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['font.family'] = 'serif'
matplotlib.rcParams['font.serif'] = ['Times New Roman']
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['axes.unicode_minus'] = False

# =========================
# Paths
# =========================
guanzhong_path = r"E:\矢量边界\gz_bountary\gz_city.shp"
china_border_path = r"E:\矢量边界\中国国界和省界的SHP格式数据\国界\bou1_4p.shp"
river_path = r"E:\矢量边界\中国三级及以上河流\R3\hyd2_4l.shp"

output_dir = r"E:\关中干旱小论文\研究区域图-图一"
os.makedirs(output_dir, exist_ok=True)

pdf_out = os.path.join(output_dir, "study_area_optimized_v2.pdf")
png_out = os.path.join(output_dir, "study_area_optimized_v2.png")

# =========================
# Extent
# =========================
lon_min, lon_max = 95, 125
lat_min, lat_max = 25, 45
extent_poly = box(lon_min, lat_min, lon_max, lat_max)

lat_mid = (lat_min + lat_max) / 2.0
d_lon_km = (lon_max - lon_min) * 111.32 * np.cos(np.radians(lat_mid))
d_lat_km = (lat_max - lat_min) * 111.32

FIG_W = 11.5
FIG_H = FIG_W * d_lat_km / d_lon_km * 0.92

# =========================
# Fully renewed color scheme
# =========================
SEA_COLOR   = "#F2F8FC"
LAND_OTHER  = "#EEEDE8"
CHINA_COLOR = "#F8F7F2"
CHINA_EDGE  = "#4A443D"

GZ_COLOR = "#EFB2AE"
GZ_EDGE  = "#BC6B64"
GZ_HALO  = "#F6D6D3"

RIVER_L1   = "#2D7FB8"
RIVER_L2   = "#5DA8D8"
RIVER_L3   = "#9ACDEA"
RIVER_TEXT = "#2A597A"

GRID_COLOR = "#C8D2DA"
TEXT_COLOR = "#262626"

# =========================
# Utilities
# =========================
def read_shp_multi_encoding(path, encodings=("utf-8", "gbk", "gb18030", "cp936")):
    last_err = None
    for enc in encodings:
        try:
            gdf = gpd.read_file(path, encoding=enc)
            print(f"[OK] Read: {os.path.basename(path)} with encoding={enc}")
            return gdf
        except Exception as e:
            last_err = e
    print(f"[WARN] fallback read without encoding: {os.path.basename(path)}")
    try:
        return gpd.read_file(path)
    except Exception as e:
        raise RuntimeError(f"Failed to read {path}, last error: {e}, prev: {last_err}")

def safe_4326(gdf):
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    else:
        gdf = gdf.to_crs("EPSG:4326")
    return gdf[gdf.geometry.notnull() & ~gdf.geometry.is_empty].copy()

def build_china_geom(china_clip):
    geom_types = set(china_clip.geom_type.unique())
    if any(t in ("Polygon", "MultiPolygon") for t in geom_types):
        return china_clip.dissolve().geometry.iloc[0]
    merged = unary_union(china_clip.geometry.values)
    polys = list(polygonize(merged))
    if not polys:
        return None
    kept = [p for p in polys if extent_poly.contains(p.representative_point())]
    return unary_union(kept) if kept else None

def normalize_level(series):
    s = series.astype(str).str.strip()
    s = s.replace({"": np.nan, "nan": np.nan, "None": np.nan})
    return pd.to_numeric(s, errors="coerce")

def longest_line(geom):
    if geom is None or geom.is_empty:
        return None
    if isinstance(geom, LineString):
        return geom
    if isinstance(geom, MultiLineString):
        geoms = [g for g in geom.geoms if g.length > 0]
        if not geoms:
            return None
        try:
            merged = linemerge(geom)
            if isinstance(merged, LineString):
                return merged
            elif isinstance(merged, MultiLineString):
                return max(list(merged.geoms), key=lambda g: g.length)
        except Exception:
            pass
        return max(geoms, key=lambda g: g.length)
    return None

def global_angle_of_line(line):
    """
    Use large-scale trend rather than local tangent.
    """
    if line is None or line.length == 0:
        return 0.0

    p0 = line.interpolate(line.length * 0.10)
    p1 = line.interpolate(line.length * 0.90)
    dx = p1.x - p0.x
    dy = p1.y - p0.y

    if abs(dx) < 1e-12 and abs(dy) < 1e-12:
        p0 = line.coords[0]
        p1 = line.coords[-1]
        dx = p1[0] - p0[0]
        dy = p1[1] - p0[1]

    ang = np.degrees(np.arctan2(dy, dx))
    if ang > 90:
        ang -= 180
    if ang < -90:
        ang += 180
    return ang

def normal_offset_by_global_angle(point, angle_deg, offset_deg=0.14, side=1):
    """
    Offset label beside river using a normal vector derived from the global angle.
    """
    theta = np.radians(angle_deg)
    nx = -np.sin(theta)
    ny = np.cos(theta)
    x = point.x + side * offset_deg * nx
    y = point.y + side * offset_deg * ny
    return x, y

def valid_label_text(x):
    if x is None:
        return False
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return False
    return True

def pick_label_side(line):
    rep = line.interpolate(line.length * 0.5)
    return 1 if rep.y < 35 else -1

def add_single_river_label(ax, geom, text, fontsize, placed, min_sep=0.8, offset_deg=0.14):
    line = longest_line(geom)
    if line is None or line.length == 0:
        return placed

    dist = line.length * 0.52
    point = line.interpolate(dist)
    angle = global_angle_of_line(line)
    side = pick_label_side(line)
    x, y = normal_offset_by_global_angle(point, angle, offset_deg=offset_deg, side=side)

    for px, py in placed:
        if ((x - px)**2 + (y - py)**2)**0.5 < min_sep:
            return placed

    ax.text(
        x, y, text,
        fontsize=fontsize,
        color=RIVER_TEXT,
        rotation=angle,
        rotation_mode='anchor',
        ha='center',
        va='center',
        zorder=10,
        path_effects=[
            pe.Stroke(linewidth=2.8, foreground="white", alpha=0.98),
            pe.Normal()
        ]
    )
    placed.append((x, y))
    return placed

def add_text_along_line_global_angle(ax, geom, text, fontsize=12, offset_deg=0.22, color=RIVER_TEXT, zorder=10):
    """
    Characters are distributed along the river line,
    but all share one global angle based on large-scale direction.
    """
    line = longest_line(geom)
    if line is None or line.length == 0:
        return

    chars = list(text)
    n = len(chars)
    if n == 0:
        return

    start = line.length * 0.20
    end = line.length * 0.80
    if end <= start:
        return

    positions = np.linspace(start, end, n)
    angle = global_angle_of_line(line)
    side = pick_label_side(line)

    for ch, dist in zip(chars, positions):
        if ch == " ":
            continue
        point = line.interpolate(dist)
        x, y = normal_offset_by_global_angle(point, angle, offset_deg=offset_deg, side=side)
        ax.text(
            x, y, ch,
            fontsize=fontsize,
            color=color,
            rotation=angle,
            rotation_mode='anchor',
            ha='center',
            va='center',
            zorder=zorder,
            path_effects=[
                pe.Stroke(linewidth=3.0, foreground="white", alpha=0.98),
                pe.Normal()
            ]
        )

def draw_scale_bar_outside(fig, x=0.79, y=0.16, w=0.15, color=TEXT_COLOR):
    ax2 = fig.add_axes([x, y, w, 0.06])
    ax2.set_axis_off()
    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, 1)
    ax2.plot([0.08, 0.92], [0.42, 0.42], color=color, lw=2.0, solid_capstyle='butt')
    ax2.plot([0.08, 0.08], [0.24, 0.60], color=color, lw=1.2)
    ax2.plot([0.92, 0.92], [0.24, 0.60], color=color, lw=1.2)
    ax2.text(0.50, 0.70, "500 km", ha='center', va='bottom', fontsize=10.4, color=color)

def draw_north_arrow_outside(fig, x=0.845, y=0.28, w=0.05, h=0.12, color=TEXT_COLOR):
    ax2 = fig.add_axes([x, y, w, h])
    ax2.set_axis_off()
    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, 1)
    ax2.annotate(
        "",
        xy=(0.5, 0.92), xytext=(0.5, 0.16),
        arrowprops=dict(arrowstyle="-|>", color=color, lw=1.55, mutation_scale=17)
    )
    ax2.text(0.5, 0.98, "N", ha='center', va='bottom', fontsize=12, fontweight='bold', color=color)

# =========================
# Read data
# =========================
china = safe_4326(read_shp_multi_encoding(china_border_path))
china_clip = gpd.clip(china, extent_poly)
china_geom = build_china_geom(china_clip)

gz = safe_4326(read_shp_multi_encoding(guanzhong_path))
gz_clip = gpd.clip(gz.dissolve(), extent_poly)

rivers = safe_4326(read_shp_multi_encoding(river_path))
rivers_clip = gpd.clip(rivers, extent_poly)

print("\n[River fields]")
print(rivers_clip.columns.tolist())
print(rivers_clip.head(3))

# =========================
# World land
# =========================
try:
    import geodatasets
    world_raw = safe_4326(gpd.read_file(geodatasets.get_path("naturalearth.land")))
    world_clip = gpd.clip(world_raw, extent_poly)
except Exception:
    world_clip = None

# =========================
# River processing
# =========================
river_level_field = "LEVEL_RIVE"
river_name_field = "NAME"

rv = rivers_clip.copy()
rv["_level_num_"] = normalize_level(rv[river_level_field])
rv["_name_raw_"] = rv[river_name_field].astype(str).str.strip()

yellow_system_names = {
    "黄河", "渭河", "汾河", "泾河", "洛河", "北洛河", "无定河", "洮河", "湟水", "大通河"
}

yangtze_system_names = {
    "长江", "通天河", "金沙江", "雅砻江", "岷江", "嘉陵江", "汉江(汉水)", "乌江", "大渡河"
}

is_yellow = rv["_name_raw_"].isin(yellow_system_names)
is_yangtze = rv["_name_raw_"].isin(yangtze_system_names)

river_yellow = rv[is_yellow & (rv["_level_num_"] <= 3)].copy()
river_yangtze = rv[is_yangtze & (rv["_level_num_"] <= 2)].copy()
river_other = rv[(~is_yellow) & (~is_yangtze) & (rv["_level_num_"] <= 2)].copy()

river_draw = pd.concat([river_yellow, river_yangtze, river_other], ignore_index=True)
river_draw = river_draw[river_draw.geometry.notnull() & ~river_draw.geometry.is_empty].copy()

river_l1 = river_draw[river_draw["_level_num_"] == 1].copy()
river_l2 = river_draw[river_draw["_level_num_"] == 2].copy()
river_l3 = river_draw[river_draw["_level_num_"] == 3].copy()

print("\n[Selected river statistics]")
print(f"Yellow-system segments (<=3): {len(river_yellow)}")
print(f"Yangtze-system segments (<=2): {len(river_yangtze)}")
print(f"Other-system segments (<=2): {len(river_other)}")
print(f"Level-1 drawn: {len(river_l1)}")
print(f"Level-2 drawn: {len(river_l2)}")
print(f"Level-3 drawn: {len(river_l3)}")

# =========================
# Labels: English where available, otherwise Chinese fallback
# =========================
river_name_en_map = {
    "黄河": "Yellow River",
    "长江": "Yangtze River",
    "渭河": "Wei River",
    "汾河": "Fen River",
    "泾河": "Jing River",
    "洛河": "Luo River",
    "北洛河": "Beiluo River",
    "无定河": "Wuding River",
    "洮河": "Tao River",
    "湟水": "Huangshui River",
    "大通河": "Datong River",
    "嘉陵江": "Jialing River",
    "岷江": "Min River",
    "汉江(汉水)": "Han River",
    "海河": "Hai River",
    "淮河": "Huai River",
    "红水河": "Hongshui River",
    "南盘江": "Nanpan River",
    "金沙江": "Jinsha River",
    "雅砻江": "Yalong River",
    "礼社江": "Lishe River",
    "怒江": "Nu River",
    "澜沧江": "Lancang River",
    "雅鲁藏布江": "Yarlung Zangbo River",
    "通天河": "Tongtian River",
    "西辽河": "Xiliao River",
    "辽河": "Liao River",
    "老哈河": "Laoha River",
    "双台子河": "Shuangtaizi River",
    "卫河(大沙河)": "Wei River",
    "沮水": "Ju River",
    "扎曲": "Za Qu",
    "伊洛瓦底江(独龙江、恩梅开江)": "Irrawaddy River",
    "约古宗列渠": "Yueguzonglie Qu",
    "淮河入江水道": "Huai River Waterway",
    "乌江": "Wu River",
    "大渡河": "Dadu River"
}

river_draw["_name_en_"] = river_draw["_name_raw_"].map(river_name_en_map)
river_draw["_label_"] = river_draw["_name_en_"].fillna(river_draw["_name_raw_"])
river_lab = river_draw[river_draw["_label_"].apply(valid_label_text)].copy()

# =========================
# Figure
# =========================
fig = plt.figure(figsize=(FIG_W, FIG_H), dpi=300)
ax = fig.add_axes([0.055, 0.08, 0.70, 0.84])

ax.set_facecolor(SEA_COLOR)
ax.set_xlim(lon_min, lon_max)
ax.set_ylim(lat_min, lat_max)
ax.set_aspect("auto")

# =========================
# Base layers: truly updated colors
# =========================
if world_clip is not None and len(world_clip) > 0:
    world_clip.plot(ax=ax, facecolor=LAND_OTHER, edgecolor="none", zorder=1)

if china_geom is not None:
    gpd.GeoDataFrame(geometry=[china_geom], crs="EPSG:4326").plot(
        ax=ax, facecolor=CHINA_COLOR, edgecolor="none", zorder=2
    )

# Guanzhong below rivers
gz_clip.plot(
    ax=ax,
    facecolor=GZ_COLOR,
    edgecolor=GZ_EDGE,
    linewidth=0.85,
    alpha=0.52,
    zorder=3
)
gz_clip.boundary.plot(
    ax=ax,
    color=GZ_EDGE,
    linewidth=1.0,
    alpha=0.72,
    zorder=3.1
)

# Rivers above Guanzhong
if len(river_l3) > 0:
    river_l3.plot(ax=ax, color=RIVER_L3, linewidth=0.42, alpha=0.96, zorder=4)

if len(river_l2) > 0:
    river_l2.plot(ax=ax, color=RIVER_L2, linewidth=0.64, alpha=0.98, zorder=5)

if len(river_l1) > 0:
    river_l1.plot(ax=ax, color=RIVER_L1, linewidth=1.05, alpha=0.99, zorder=6)

# China border
china_clip.plot(ax=ax, facecolor="none", edgecolor=CHINA_EDGE, linewidth=0.82, zorder=7)

# Guanzhong label
try:
    c = gz_clip.geometry.iloc[0].representative_point()
    ax.text(
        c.x, c.y + 0.35, "Guanzhong Plain",
        fontsize=11.8,
        color=GZ_EDGE,
        ha='center',
        va='bottom',
        zorder=10,
        path_effects=[pe.Stroke(linewidth=3.0, foreground="white"), pe.Normal()]
    )
except Exception:
    pass

# =========================
# River labels
# =========================
placed = []

for label_name, sub in river_lab.groupby("_label_"):
    geoms = [g for g in sub.geometry if g is not None and not g.is_empty]
    if not geoms:
        continue

    merged_geom = unary_union(geoms)

    if label_name == "Yellow River":
        add_text_along_line_global_angle(ax, merged_geom, "Yellow River", fontsize=11.8, offset_deg=0.24)
    elif label_name == "Yangtze River":
        add_text_along_line_global_angle(ax, merged_geom, "Yangtze River", fontsize=11.8, offset_deg=0.24)
    else:
        levels = sub["_level_num_"].dropna()
        lv = int(levels.min()) if len(levels) > 0 else 2

        if lv == 1:
            fs = 9.7
            sep = 0.95
            offset = 0.15
        elif lv == 2:
            fs = 8.8
            sep = 0.78
            offset = 0.13
        else:
            fs = 8.0
            sep = 0.62
            offset = 0.11

        placed = add_single_river_label(
            ax=ax,
            geom=merged_geom,
            text=label_name,
            fontsize=fs,
            placed=placed,
            min_sep=sep,
            offset_deg=offset
        )

# =========================
# Grid and axes
# =========================
ax.grid(
    True,
    linestyle=(0, (2.5, 3.0)),
    linewidth=0.28,
    color=GRID_COLOR,
    alpha=0.22,
    zorder=0
)

ax.set_xticks(range(95, 126, 5))
ax.set_yticks(range(25, 46, 5))
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{int(v)}°E"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{int(v)}°N"))

ax.tick_params(axis='both', labelsize=11.4, length=4.0, width=0.75, direction='out', pad=4)
ax.set_xlabel("Longitude", fontsize=12.4, labelpad=6, color=TEXT_COLOR)
ax.set_ylabel("Latitude", fontsize=12.4, labelpad=6, color=TEXT_COLOR)

for sp in ax.spines.values():
    sp.set_linewidth(0.88)
    sp.set_color(CHINA_EDGE)

# =========================
# Legend outside
# =========================
legend_handles = [
    mpatches.Patch(facecolor=GZ_COLOR, edgecolor=GZ_EDGE, linewidth=0.8, label="Guanzhong Plain"),
    mpatches.Patch(facecolor=CHINA_COLOR, edgecolor=CHINA_EDGE, linewidth=0.6, label="China"),
    mpatches.Patch(facecolor=LAND_OTHER, edgecolor="none", label="Other countries"),
    mpatches.Patch(facecolor=SEA_COLOR, edgecolor="none", label="Sea"),
    Line2D([0], [0], color=RIVER_L1, lw=1.2, label="Level-1 rivers"),
    Line2D([0], [0], color=RIVER_L2, lw=0.95, label="Level-2 rivers"),
    Line2D([0], [0], color=RIVER_L3, lw=0.8, label="Level-3 rivers (Yellow River system)")
]

leg = fig.legend(
    handles=legend_handles,
    loc="upper left",
    bbox_to_anchor=(0.78, 0.82),
    frameon=True,
    framealpha=0.98,
    edgecolor="#D3CEC7",
    facecolor="white",
    fontsize=10.1,
    borderpad=0.88,
    labelspacing=0.62,
    handlelength=2.1,
    handletextpad=0.8
)
leg.get_frame().set_linewidth(0.65)

draw_scale_bar_outside(fig, x=0.79, y=0.16, w=0.15, color=TEXT_COLOR)
draw_north_arrow_outside(fig, x=0.845, y=0.28, w=0.05, h=0.12, color=TEXT_COLOR)

# =========================
# Save
# =========================
plt.savefig(pdf_out, format="pdf", dpi=300, bbox_inches="tight", facecolor="white")
plt.savefig(png_out, format="png", dpi=600, bbox_inches="tight", facecolor="white")
plt.show()

print(f"Saved PDF: {pdf_out}")
print(f"Saved PNG: {png_out}")
